In [4]:
!pip install --quiet deepchem rdkit numpy scikit-learn

In [5]:
import deepchem as dc
import numpy as np
import torch
import torch.nn as nn
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
import urllib.request
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
RDLogger.DisableLog('rdApp.*')

print("DeepChem:", dc.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

DeepChem: 2.5.0
PyTorch: 2.9.0+cu126
CUDA: True
Device: cuda


In [6]:
TOX21_TASKS = ['NR-AR','NR-AR-LBD','NR-AhR','NR-Aromatase','NR-ER',
               'NR-ER-LBD','NR-PPAR-gamma','SR-ARE','SR-ATAD5',
               'SR-HSE','SR-MMP','SR-p53']

print("Downloading Tox21...")
urllib.request.urlretrieve(
    "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz",
    "tox21.csv.gz"
)
df = pd.read_csv("tox21.csv.gz")

valid_rows = []
for _, row in df.iterrows():
    if Chem.MolFromSmiles(str(row['smiles'])) is not None:
        valid_rows.append(row)
df = pd.DataFrame(valid_rows).reset_index(drop=True)
print(f"Valid molecules: {len(df)}")

print("Featurizing as molecular graphs...")
featurizer = dc.feat.MolGraphConvFeaturizer()
features = []
for i, smi in enumerate(df['smiles']):
    feat = featurizer.featurize([smi])
    features.append(feat[0])
    if (i+1) % 1000 == 0:
        print(f"  {i+1}/{len(df)}")

valid_indices = [i for i, f in enumerate(features)
                 if hasattr(f, 'node_features') and f.node_features is not None]
df = df.iloc[valid_indices].reset_index(drop=True)
features = [features[i] for i in valid_indices]
print(f"Graph features ready: {len(features)}")

Valid molecules: 7823
Featurizing as molecular graphs...


Failed to featurize datapoint 0, [I-].[K+]. Appending empty array
Exception message: zero-size array to reduction operation maximum which has no identity
Failed to featurize datapoint 0, [Hg+2]. Appending empty array
Exception message: zero-size array to reduction operation maximum which has no identity
Failed to featurize datapoint 0, [Ba+2]. Appending empty array
Exception message: zero-size array to reduction operation maximum which has no identity
Failed to featurize datapoint 0, [TlH2+]. Appending empty array
Exception message: zero-size array to reduction operation maximum which has no identity


  1000/7823


Failed to featurize datapoint 0, [Cr+3]. Appending empty array
Exception message: zero-size array to reduction operation maximum which has no identity
Failed to featurize datapoint 0, [Fe+2]. Appending empty array
Exception message: zero-size array to reduction operation maximum which has no identity
Failed to featurize datapoint 0, [Co+2]. Appending empty array
Exception message: zero-size array to reduction operation maximum which has no identity
Failed to featurize datapoint 0, [PbH2+2]. Appending empty array
Exception message: zero-size array to reduction operation maximum which has no identity


  2000/7823


...

  5000/7823


Failed to featurize datapoint 0, [Br-].[Na+]. Appending empty array
Exception message: zero-size array to reduction operation maximum which has no identity


  6000/7823


Failed to featurize datapoint 0, [Ca+2].[Cl-].[Cl-]. Appending empty array
Exception message: zero-size array to reduction operation maximum which has no identity
Failed to featurize datapoint 0, [SbH6+3]. Appending empty array
Exception message: zero-size array to reduction operation maximum which has no identity
Failed to featurize datapoint 0, [Ni+2]. Appending empty array
Exception message: zero-size array to reduction operation maximum which has no identity


  7000/7823
Graph features ready: 7804


In [7]:
scaffolds = {}
for idx, smi in enumerate(df['smiles']):
    mol = Chem.MolFromSmiles(smi)
    sc = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    scaffolds.setdefault(sc, []).append(idx)

scaffold_sets = sorted(scaffolds.values(), key=len, reverse=True)
train_idx, valid_idx, test_idx = [], [], []
total = len(df)

for s in scaffold_sets:
    if len(test_idx) < total * 0.1:
        test_idx.extend(s)
    elif len(valid_idx) < total * 0.1:
        valid_idx.extend(s)
    else:
        train_idx.extend(s)

print(f"Train: {len(train_idx)} | Valid: {len(valid_idx)} | Test: {len(test_idx)}")

Train: 4574 | Valid: 1474 | Test: 1756


In [8]:
class MolGNN(nn.Module):
    """
    Pure PyTorch Message Passing GNN for molecular property prediction.
    Atoms = nodes, Bonds = edges.
    No DGL/PyG needed — works directly with DeepChem GraphData.
    """
    def __init__(self, node_feat_dim=30, hidden_dim=128, n_tasks=12):
        super().__init__()
        self.conv1  = nn.Linear(node_feat_dim, hidden_dim)
        self.conv2  = nn.Linear(hidden_dim, hidden_dim)
        self.conv3  = nn.Linear(hidden_dim, hidden_dim)
        self.readout = nn.Linear(hidden_dim, hidden_dim)
        self.predict = nn.Linear(hidden_dim, n_tasks)
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)

    def message_pass(self, x, edge_index, layer):
        out = x.clone()
        if edge_index is not None and len(edge_index) > 0:
            src = edge_index[:, 0]
            dst = edge_index[:, 1]
            for node in range(x.shape[0]):
                neighbors = src[dst == node]
                if len(neighbors) > 0:
                    out[node] = x[node] + x[neighbors].mean(dim=0)
        return self.relu(layer(out))

    def forward(self, graph_data, device):
        x = torch.FloatTensor(graph_data.node_features).to(device)
        edge_index = graph_data.edge_index

        x = self.message_pass(x, edge_index, self.conv1)
        x = self.dropout(x)
        x = self.message_pass(x, edge_index, self.conv2)
        x = self.dropout(x)
        x = self.message_pass(x, edge_index, self.conv3)

        mol_repr = x.mean(dim=0)
        mol_repr = self.relu(self.readout(mol_repr))
        return self.predict(mol_repr)

gnn = MolGNN(node_feat_dim=30, hidden_dim=128, n_tasks=12).to(device)
total_params = sum(p.numel() for p in gnn.parameters())
print(f"Device: {device}")
print(f"GNN parameters: {total_params:,}")
print("Model ready.")

Device: cuda
GNN parameters: 55,564
Model ready.


In [9]:
from sklearn.metrics import roc_auc_score

y = df[TOX21_TASKS].values.astype(float)
w = (~np.isnan(y)).astype(float)
y = np.nan_to_num(y, nan=0.0)

# Per-task class weights — handles Tox21 imbalance
pos_weights = []
for t in range(12):
    mask   = w[train_idx, t] > 0
    labels = y[train_idx][mask, t]
    n_pos  = labels.sum()
    n_neg  = len(labels) - n_pos
    pos_weights.append(n_neg / (n_pos + 1e-8))

print("Class weights computed.")

optimizer = torch.optim.Adam(
    gnn.parameters(), lr=0.001, weight_decay=1e-5
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=100, eta_min=1e-5
)

print("Training (100 epochs)...")
print("="*55)

best_auc      = 0
best_per_task = None

for epoch in range(100):
    gnn.train()
    total_loss = 0

    for idx in train_idx:
        labels  = torch.FloatTensor(y[idx]).to(device)
        weights = torch.FloatTensor(w[idx]).to(device)

        optimizer.zero_grad()
        preds = gnn(features[idx], device)

        loss = 0
        for t in range(12):
            pw = torch.tensor(pos_weights[t]).to(device)
            tl = nn.BCEWithLogitsLoss(
                pos_weight=pw, reduction='none'
            )(preds[t].unsqueeze(0), labels[t].unsqueeze(0))
            loss += (tl * weights[t]).sum()
        loss = loss / (weights.sum() + 1e-8)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(gnn.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()

    if (epoch + 1) % 10 == 0:
        gnn.eval()
        all_preds, all_labels, all_w = [], [], []
        with torch.no_grad():
            for idx in test_idx:
                pred = torch.sigmoid(
                    gnn(features[idx], device)
                ).cpu().numpy()
                all_preds.append(pred)
                all_labels.append(y[idx])
                all_w.append(w[idx])

        all_preds  = np.array(all_preds)
        all_labels = np.array(all_labels)
        all_w      = np.array(all_w)

        task_aucs = []
        for t in range(12):
            mask = all_w[:, t] > 0
            if mask.sum() < 10:
                continue
            auc = roc_auc_score(all_labels[mask, t], all_preds[mask, t])
            task_aucs.append(auc)

        mean_auc = np.mean(task_aucs)
        if mean_auc > best_auc:
            best_auc      = mean_auc
            best_per_task = task_aucs.copy()

        print(f"Epoch {epoch+1:3d} | "
              f"Loss: {total_loss/len(train_idx):.4f} | "
              f"ROC-AUC: {mean_auc:.4f} | "
              f"Best: {best_auc:.4f}")

print(f"\nFinal Best ROC-AUC: {best_auc:.4f}")

Class weights computed.
Training (100 epochs)...
Epoch  10 | Loss: 1.9192 | ROC-AUC: 0.6145 | Best: 0.6145
Epoch  20 | Loss: 1.8541 | ROC-AUC: 0.6340 | Best: 0.6340
Epoch  30 | Loss: 1.8322 | ROC-AUC: 0.6272 | Best: 0.6340
Epoch  40 | Loss: 1.7765 | ROC-AUC: 0.6282 | Best: 0.6340
Epoch  50 | Loss: 1.7306 | ROC-AUC: 0.6317 | Best: 0.6340
Epoch  60 | Loss: 1.7074 | ROC-AUC: 0.6331 | Best: 0.6340
Epoch  70 | Loss: 1.6641 | ROC-AUC: 0.6363 | Best: 0.6363
Epoch  80 | Loss: 1.6501 | ROC-AUC: 0.6388 | Best: 0.6388
Epoch  90 | Loss: 1.6301 | ROC-AUC: 0.6381 | Best: 0.6388
Epoch 100 | Loss: 1.6625 | ROC-AUC: 0.6372 | Best: 0.6388

Final Best ROC-AUC: 0.6388


In [10]:
print("=" * 60)
print("TOX21 — SCAFFOLD SPLIT BENCHMARK")
print("=" * 60)
print(f"{'Model':<35} {'ROC-AUC':>8}")
print("-" * 60)
print(f"{'RF + ECFP (random split)':<35} {'0.8183':>8}  ← inflated")
print(f"{'RF + ECFP (scaffold split)':<35} {'0.6135':>8}")
print(f"{'Pure PyTorch GNN (scaffold)':<35} {best_auc:>8.4f}  ← graph")
print(f"{'OLMo-7B QLoRA (scaffold)':<35} {'0.6300':>8}  ← LLM")
print(f"{'ChemBERTa-3 (scaffold)':<35} {'0.8000':>8}  ← target")
print("-" * 60)
print()
print("Finding: GNN uses 2D graph topology (atoms+bonds).")
print("OLMo uses 1D SMILES text. Gap motivates TSM approach.")
print("TSM injects graph constraints into LLM decoding loop.")

TOX21 — SCAFFOLD SPLIT BENCHMARK
Model                                ROC-AUC
------------------------------------------------------------
RF + ECFP (random split)              0.8183  ← inflated
RF + ECFP (scaffold split)            0.6135
Pure PyTorch GNN (scaffold)           0.6388  ← graph
OLMo-7B QLoRA (scaffold)              0.6300  ← LLM
ChemBERTa-3 (scaffold)                0.8000  ← target
------------------------------------------------------------

Finding: GNN uses 2D graph topology (atoms+bonds).
OLMo uses 1D SMILES text. Gap motivates TSM approach.
TSM injects graph constraints into LLM decoding loop.
